# TFT Scouting — Notebook 3: Meta Analysis

**Objetivo:** A partir del NDJSON crudo, extraer:
1. Win rates de composiciones (top-4 rate por trait dominante)
2. Items más frecuentes por carry (`itemNames`)
3. Variantes: mismo comp core → distintos units/items → distintos placements
4. Resumen por comp para el prompt de IA

**Nota:** Los augments no están disponibles en match-v1 para Set 16 aún.
Se añadirán cuando Riot actualice el endpoint.

In [1]:
import json
import pandas as pd
import numpy as np
from collections import Counter

# ── Cargar datos ──────────────────────────────────────────────────────────────
matches = []
with open('matches_raw.ndjson', 'r') as f:
    for line in f:
        matches.append(json.loads(line))

print(f'Partidas cargadas: {len(matches)}')

Partidas cargadas: 2745


In [2]:
# ── Aplanar participantes ─────────────────────────────────────────────────────

rows = []
for match in matches:
    match_id     = match['metadata']['match_id']
    game_datetime = match['info'].get('game_datetime', 0)
    game_version  = match['info'].get('game_version', '')
    for p in match['info']['participants']:
        rows.append({
            'match_id':                match_id,
            'game_datetime':           game_datetime,
            'game_version':            game_version,
            'puuid':                   p.get('puuid'),
            'placement':               p.get('placement'),
            'level':                   p.get('level'),
            'last_round':              p.get('last_round'),
            'gold_left':               p.get('gold_left', 0),
            'total_damage_to_players': p.get('total_damage_to_players', 0),
            'players_eliminated':      p.get('players_eliminated', 0),
            'traits':                  p.get('traits', []),
            'units':                   p.get('units', []),
        })

df = pd.DataFrame(rows)
df['top4'] = df['placement'] <= 4
df['win']  = df['placement'] == 1

print(f'Participantes totales: {len(df)}')
print(f'Partidas únicas:       {df["match_id"].nunique()}')
print(f'Top-4 rate global:     {df["top4"].mean():.1%}  (esperado ~50%)')
df.head(2)

Participantes totales: 21883
Partidas únicas:       2745
Top-4 rate global:     50.3%  (esperado ~50%)


,match_id,game_datetime,game_version,puuid,placement,level,last_round,gold_left,total_damage_to_players,players_eliminated,traits,units,top4,win
0,EUW1_7761291413,1772515096867,Linux Version 16.4.748.0682 (Feb 20 2026/12:43...,UyM8O0EJnOuxoE9IR3JUkQWjK076_cW_Ku9OLOCHHK-kek...,3,9,35,0,110,1,"[{'name': 'TFT16_Brawler', 'num_units': 2, 'st...","[{'character_id': 'TFT16_Shen', 'itemNames': [...",True,False
1,EUW1_7761291413,1772515096867,Linux Version 16.4.748.0682 (Feb 20 2026/12:43...,cIS7pCMhM9ugJZcxVqpKmFxJH3qNNX4JKefGin6ZKa91_N...,8,8,23,0,11,0,"[{'name': 'TFT16_Brawler', 'num_units': 2, 'st...","[{'character_id': 'TFT16_KogMaw', 'itemNames':...",False,False


In [3]:
# ── 1. Traits activos por participante ────────────────────────────────────────

def get_active_traits(traits):
    return [
        (t['name'], t.get('tier_current', 0), t.get('num_units', 0))
        for t in traits
        if t.get('tier_current', 0) > 0
    ]

def get_dominant_trait(traits):
    active = get_active_traits(traits)
    if not active:
        return 'Unknown'
    return max(active, key=lambda x: x[2])[0]

def get_trait_signature(traits, top_n=3):
    active = sorted(get_active_traits(traits), key=lambda x: x[2], reverse=True)
    return tuple(t[0] for t in active[:top_n])

df['dominant_trait']  = df['traits'].apply(get_dominant_trait)
df['trait_signature'] = df['traits'].apply(get_trait_signature)

print('Muestra de traits detectados:')
df[['dominant_trait', 'trait_signature', 'placement', 'top4']].head(10)

Muestra de traits detectados:


,dominant_trait,trait_signature,placement,top4
0,TFT16_Ionia,"(TFT16_Ionia, TFT16_Brawler, TFT16_Juggernaut)",3,True
1,TFT16_Juggernaut,"(TFT16_Juggernaut, TFT16_Void, TFT16_Brawler)",8,False
2,TFT16_Zaun,"(TFT16_Zaun, TFT16_Brawler, TFT16_Invoker)",4,True
3,TFT16_Piltover,"(TFT16_Piltover, TFT16_Defender, TFT16_Gunslin...",2,True
4,TFT16_Defender,"(TFT16_Defender, TFT16_Magus, TFT16_Piltover)",7,False
5,TFT16_Demacia,"(TFT16_Demacia, TFT16_Juggernaut, TFT16_Defender)",5,False
6,TFT16_Ionia,"(TFT16_Ionia, TFT16_Brawler, TFT16_Defender)",1,True
7,TFT16_Defender,"(TFT16_Defender, TFT16_Juggernaut, TFT16_Magus)",6,False
8,TFT16_Ionia,"(TFT16_Ionia, TFT16_Brawler, TFT16_Defender)",6,False
9,TFT16_Freljord,"(TFT16_Freljord, TFT16_Zaun, TFT16_Defender)",4,True


In [4]:
# ── 2. Top-4 rate por trait dominante ─────────────────────────────────────────

trait_stats = (
    df.groupby('dominant_trait')
    .agg(
        games         = ('placement', 'count'),
        top4_rate     = ('top4', 'mean'),
        avg_placement = ('placement', 'mean'),
        win_rate      = ('win', 'mean')
    )
    .query('games >= 20')
    .sort_values('top4_rate', ascending=False)
    .round(3)
)

print('=== Top traits por top-4 rate ===')
print(trait_stats.head(20).to_string())

=== Top traits por top-4 rate ===
                  games  top4_rate  avg_placement  win_rate
dominant_trait                                             
Unknown             254      0.768          2.760     0.563
TFT16_Shurima       137      0.759          2.964     0.350
TFT16_Warden        737      0.586          4.213     0.098
TFT16_Slayer        275      0.585          4.178     0.069
TFT16_Gunslinger     33      0.576          3.909     0.242
TFT16_Sorcerer     1068      0.567          4.135     0.172
TFT16_Juggernaut    778      0.553          4.350     0.104
TFT4_Elderwood       22      0.545          4.318     0.182
TFT16_Void          691      0.544          4.207     0.158
TFT16_Brawler      1123      0.527          4.408     0.109
TFT16_Vanquisher    112      0.527          4.250     0.116
TFT16_Freljord     1338      0.519          4.406     0.121
TFT16_Rapidfire      31      0.516          4.323     0.161
TFT16_Targon        442      0.514          4.407     0.113
TFT16_

In [5]:
# ── 3. Items más frecuentes por carry ─────────────────────────────────────────
# Usa itemNames (nombre del campo en Set 16)
# Carry = unit con 2+ items

def get_carries(units):
    return [
        (u['character_id'], u.get('itemNames', []), u.get('rarity', 0), u.get('tier', 1))
        for u in units
        if len(u.get('itemNames', [])) >= 2
    ]

carry_item_data = []
for _, row in df[df['top4']].iterrows():
    for champ, items, rarity, tier in get_carries(row['units']):
        for item in items:
            carry_item_data.append({
                'champion':  champ,
                'item':      item,
                'tier':      tier,
                'placement': row['placement']
            })

print(f'Registros carry-item en top-4: {len(carry_item_data)}')

if len(carry_item_data) == 0:
    print('Sin carries con 2+ items — revisar datos')
else:
    df_items = pd.DataFrame(carry_item_data)

    carry_items = (
        df_items.groupby(['champion', 'item'])
        .agg(frequency=('item', 'count'))
        .reset_index()
        .sort_values(['champion', 'frequency'], ascending=[True, False])
        .groupby('champion')
        .head(3)
    )

    top_carries = (
        df_items.groupby('champion')
        .size()
        .sort_values(ascending=False)
        .head(15)
        .index
    )

    print('\n=== Top items por carry más jugado (en top-4) ===')
    print(carry_items[carry_items['champion'].isin(top_carries)].to_string())

Registros carry-item en top-4: 143846

=== Top items por carry más jugado (en top-4) ===
              champion                         item  frequency
888        TFT16_Braum          TFT_Item_Redemption        447
861        TFT16_Braum         TFT_Item_BrambleVest        360
887        TFT16_Braum             TFT_Item_RedBuff        359
2315     TFT16_Kindred   TFT_Item_GuinsoosRageblade        753
2333     TFT16_Kindred    TFT_Item_RunaansHurricane        494
2321     TFT16_Kindred   TFT_Item_MadredsBloodrazor        261
2793      TFT16_Lucian         TFT_Item_LastWhisper        592
2809      TFT16_Lucian       TFT_Item_SpearOfShojin        511
2790      TFT16_Lucian        TFT_Item_InfinityEdge        492
3028         TFT16_Mel     TFT_Item_JeweledGauntlet        650
3030         TFT16_Mel           TFT_Item_Leviathan        625
3042         TFT16_Mel       TFT_Item_SpearOfShojin        373
3635        TFT16_Ornn          TFT_Item_Redemption        215
3640        TFT16_Ornn    TFT

In [6]:
# ── 4. Comp core y variantes ──────────────────────────────────────────────────
# Core = top 3 units con mayor tier (2★ o 3★)
# Si hay empate en tier, desempata por rareza (coste)

def get_comp_core(units, top_n=3):
    upgraded = sorted(
        [u for u in units if u.get('tier', 1) >= 2],
        key=lambda x: (x.get('tier', 1), x.get('rarity', 0)),
        reverse=True
    )
    return frozenset(u['character_id'] for u in upgraded[:top_n])

df['comp_core']     = df['units'].apply(get_comp_core)
df['comp_core_str'] = df['comp_core'].apply(lambda x: ' | '.join(sorted(x)))

# Comps con suficiente muestra
core_counts = df['comp_core_str'].value_counts()
valid_cores = core_counts[core_counts >= 15].index

print(f'Comp cores con 15+ partidas: {len(valid_cores)}')
print('\nTop 15 comp cores más jugados:')
print(core_counts.head(15).to_string())

Comp cores con 15+ partidas: 241

Top 15 comp cores más jugados:
comp_core_str
TFT16_Leona | TFT16_Malzahar | TFT16_Zoe            486
TFT16_Annie | TFT16_AnnieTibbers | TFT16_Sylas      299
                                                    256
TFT16_Ahri | TFT16_Wukong | TFT16_Yone              221
TFT16_Braum | TFT16_Lissandra | TFT16_Seraphine     156
TFT16_Ambessa | TFT16_BelVeth | TFT16_RiftHerald    145
TFT16_Kindred | TFT16_Lucian | TFT16_Shyvana        129
TFT16_Jinx | TFT16_Loris | TFT16_Sejuani            124
TFT16_Ahri | TFT16_Kennen | TFT16_Wukong            124
TFT16_Kaisa | TFT16_RiftHerald | TFT16_Wukong       116
TFT16_Ahri | TFT16_Kennen | TFT16_Yone              110
TFT16_Jinx | TFT16_Singed | TFT16_Warwick            94
TFT16_Azir | TFT16_Ryze | TFT16_Sylas                94
TFT16_Garen | TFT16_Lux | TFT16_Taric                91
TFT16_Singed | TFT16_Warwick | TFT16_Wukong          90


In [7]:
# ── 5. Stats por comp core ────────────────────────────────────────────────────

comp_stats = (
    df[df['comp_core_str'].isin(valid_cores)]
    .groupby('comp_core_str')
    .agg(
        games         = ('placement', 'count'),
        top4_rate     = ('top4', 'mean'),
        avg_placement = ('placement', 'mean'),
        win_rate      = ('win', 'mean'),
        avg_level     = ('level', 'mean')
    )
    .sort_values('top4_rate', ascending=False)
    .round(3)
)

print('=== Top comp cores por top-4 rate ===')
print(comp_stats.head(15).to_string())

=== Top comp cores por top-4 rate ===
                                                   games  top4_rate  avg_placement  win_rate  avg_level
comp_core_str                                                                                          
TFT16_Aatrox | TFT16_Fiddlesticks | TFT16_Shyvana     29      1.000          2.034     0.276      9.241
TFT16_Ahri | TFT16_Kennen | TFT16_Sett                18      1.000          2.167     0.278      8.889
TFT16_Fiddlesticks | TFT16_Mel | TFT16_Shyvana        19      1.000          2.421     0.316      9.053
TFT16_Azir | TFT16_Ornn | TFT16_Zilean                20      1.000          1.450     0.750      9.650
TFT16_Azir | TFT16_Brock | TFT16_Ryze                 17      1.000          1.588     0.529      9.529
TFT16_Ryze | TFT16_Sett | TFT16_Volibear              15      1.000          1.867     0.333      9.333
TFT16_Kindred | TFT16_Ornn | TFT16_TahmKench          17      1.000          2.118     0.471      9.294
TFT16_Kindred | TFT16_Luci

In [8]:
# ── 6. Generar comp_summaries para el prompt de IA ────────────────────────────

comp_summaries = {}

for core_str in valid_cores:
    subset    = df[df['comp_core_str'] == core_str]
    top4_sub  = subset[subset['top4']]

    # Stats generales
    top4_rate     = subset['top4'].mean()
    avg_placement = subset['placement'].mean()
    n_games       = len(subset)

    # Traits más frecuentes en top-4
    trait_counter = Counter()
    for traits in top4_sub['traits']:
        for t in traits:
            if t.get('tier_current', 0) > 0:
                trait_counter[t['name']] += 1
    top_traits = trait_counter.most_common(6)

    # Items más frecuentes en carries de top-4
    item_counter = Counter()
    for units in top4_sub['units']:
        for u in units:
            if len(u.get('itemNames', [])) >= 2:
                for item in u['itemNames']:
                    item_counter[f"{u['character_id']}:{item}"] += 1
    top_carry_items = item_counter.most_common(9)

    # Nivel promedio en top-4 vs bottom-4 (señal de cuándo hacer fast9)
    avg_level_top4   = top4_sub['level'].mean()
    avg_level_bottom = subset[~subset['top4']]['level'].mean()

    comp_summaries[core_str] = {
        'core_units':       core_str.split(' | '),
        'n_games':          n_games,
        'top4_rate':        round(top4_rate, 3),
        'avg_placement':    round(avg_placement, 2),
        'win_rate':         round(subset['win'].mean(), 3),
        'avg_level_top4':   round(avg_level_top4, 1) if not pd.isna(avg_level_top4) else None,
        'avg_level_bottom': round(avg_level_bottom, 1) if not pd.isna(avg_level_bottom) else None,
        'top_traits':       top_traits,
        'top_carry_items':  top_carry_items,
    }

# Ver la mejor comp
best = max(comp_summaries, key=lambda x: comp_summaries[x]['top4_rate'])
print(f'Mejor comp core: {best}')
print(json.dumps(comp_summaries[best], indent=2, default=str))

Mejor comp core: TFT16_Aatrox | TFT16_Fiddlesticks | TFT16_Shyvana
{
  "core_units": [
    "TFT16_Aatrox",
    "TFT16_Fiddlesticks",
    "TFT16_Shyvana"
  ],
  "n_games": 29,
  "top4_rate": 1.0,
  "avg_placement": 2.03,
  "win_rate": 0.276,
  "avg_level_top4": 9.2,
  "avg_level_bottom": null,
  "top_traits": [
    [
      "TFT16_AatroxUnique",
      29
    ],
    [
      "TFT16_DarkinWeapon",
      29
    ],
    [
      "TFT16_Harvester",
      29
    ],
    [
      "TFT16_ShyvanaUnique",
      29
    ],
    [
      "TFT16_Juggernaut",
      28
    ],
    [
      "TFT16_Slayer",
      28
    ]
  ],
  "top_carry_items": [
    [
      "TFT16_BelVeth:TFT_Item_Quicksilver",
      23
    ],
    [
      "TFT16_Aatrox:TFT_Item_Bloodthirster",
      19
    ],
    [
      "TFT16_Aatrox:TFT_Item_UnstableConcoction",
      15
    ],
    [
      "TFT16_Ambessa:TFT_Item_UnstableConcoction",
      14
    ],
    [
      "TFT16_Swain:TFT_Item_SpectralGauntlet",
      12
    ],
    [
      "TFT16_BelVe

In [9]:
# ── 7. Guardar todo ───────────────────────────────────────────────────────────

with open('comp_summaries.json', 'w') as f:
    json.dump(comp_summaries, f, indent=2, default=str)

trait_stats.to_csv('trait_stats.csv')
comp_stats.to_csv('comp_stats.csv')
if len(carry_item_data) > 0:
    carry_items.to_csv('carry_items.csv', index=False)

print(f'comp_summaries.json — {len(comp_summaries)} comps')
print('trait_stats.csv')
print('comp_stats.csv')
print('carry_items.csv')

comp_summaries.json — 241 comps
trait_stats.csv
comp_stats.csv
carry_items.csv


## Siguiente paso: Guide Generator

Con `comp_summaries.json` tienes el insumo para el prompt a Claude API:

```python
import anthropic, json

with open('comp_summaries.json') as f:
    summaries = json.load(f)

# Coger la mejor comp
best_core = max(summaries, key=lambda x: summaries[x]['top4_rate'])
comp = summaries[best_core]

client = anthropic.Anthropic()

prompt = f"""
Eres un experto en TFT Set 16 (Lore & Legends) con rango Maestro en EUW.
Basándote en datos reales de {comp['n_games']} partidas de jugadores Challenger/GM de EUW,
genera una guía completa para esta composición.

CORE UNITS: {comp['core_units']}
STATS:
- Top-4 rate: {comp['top4_rate']*100:.1f}%
- Win rate: {comp['win_rate']*100:.1f}%  
- Placement promedio: {comp['avg_placement']}
- Nivel promedio en top-4: {comp['avg_level_top4']}
- Traits más activos en top-4: {comp['top_traits']}
- Items más frecuentes en carries: {comp['top_carry_items']}

Genera en JSON con esta estructura:
{{
  "nombre": "nombre corto de la comp",
  "descripcion": "2-3 líneas explicando la comp",
  "dificultad": "Baja/Media/Alta",
  "gameplan": {{
    "early": "qué hacer en stages 1-2",
    "mid": "qué hacer en stage 3, cuándo commitear",
    "late": "cómo rematar en stage 4+"
  }},
  "carries": [{{"unidad": "", "items_ideales": [], "items_alternativos": []}}],
  "variantes": [
    {{"condicion": "si encuentras X", "ajuste": "haz Y"}}
  ],
  "cuando_pivotar": ["señal 1", "señal 2"]
}}

Responde SOLO con el JSON, sin texto adicional.
"""

response = client.messages.create(
    model='claude-sonnet-4-20250514',
    max_tokens=1500,
    messages=[{'role': 'user', 'content': prompt}]
)

guia = json.loads(response.content[0].text)
print(json.dumps(guia, indent=2, ensure_ascii=False))
```